# 03 - Transfer Learning: PyTorch y TensorFlow

Transfer Learning es la técnica más poderosa en deep learning práctico.
En lugar de entrenar desde cero, tomamos un modelo pre-entrenado en ImageNet
(1.2M imágenes, 1000 clases) y lo adaptamos a nuestro problema.

## Contenido
1. Qué es Transfer Learning y por qué funciona
2. Estrategias: Feature extraction vs Fine-tuning
3. Transfer Learning con PyTorch (MobileNetV3)
4. Transfer Learning con TensorFlow/Keras (EfficientNet)
5. Comparación y mejores prácticas
6. Usando MLForge para automatizar todo

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import matplotlib.pyplot as plt
import numpy as np
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Por Qué Funciona Transfer Learning

Un modelo entrenado en ImageNet ya aprendió:
- **Capas tempranas**: bordes, texturas, colores
- **Capas medias**: partes de objetos (ojos, ruedas, pétalos)
- **Capas profundas**: objetos completos

Estos patrones son **universales** — sirven para casi cualquier tarea de visión.
Solo necesitamos cambiar la última capa para nuestras clases.

```
ImageNet model:  [Features] → [Classifier: 1000 clases]
                      ↓
Nuestro modelo:  [Features (congelados)] → [Nuevo Classifier: N clases]
```

## 2. Estrategias de Transfer Learning

### Feature Extraction (congelar todo)
- Congelar todas las capas pre-entrenadas
- Solo entrenar el nuevo clasificador
- Rápido (pocos parámetros a entrenar)
- Ideal cuando tienes pocos datos (<1000 imágenes)

### Fine-tuning (descongelar parcialmente)
- Descongelar las últimas capas
- Entrenar con learning rate bajo
- Mejor accuracy pero necesita más datos
- Risk de overfitting si tienes pocos datos

### Full Fine-tuning (descongelar todo)
- Entrenar todas las capas con LR muy bajo
- Solo si tienes muchos datos (>10K imágenes)
- El mejor resultado posible

In [ ]:
# Dataset: Flowers102 (exacto al que usa MLForge por defecto)
# 102 clases de flores, ~8000 imágenes

transform_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

transform_val = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

trainset = torchvision.datasets.Flowers102(
    root='./data', split='train', download=True, transform=transform_train
)
valset = torchvision.datasets.Flowers102(
    root='./data', split='val', download=True, transform=transform_val
)
testset = torchvision.datasets.Flowers102(
    root='./data', split='test', download=True, transform=transform_val
)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True, num_workers=2)
valloader = torch.utils.data.DataLoader(valset, batch_size=32, shuffle=False, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False, num_workers=2)

print(f"Train: {len(trainset)}, Val: {len(valset)}, Test: {len(testset)}")
print(f"Clases: 102 tipos de flores")

## 3. Transfer Learning con PyTorch: MobileNetV3

In [ ]:
def create_mobilenet_transfer(num_classes, freeze=True):
    """Crear MobileNetV3 para transfer learning."""
    # Descargar modelo pre-entrenado en ImageNet
    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    
    if freeze:
        # Congelar todas las capas (no se actualizan durante training)
        for param in model.parameters():
            param.requires_grad = False
    
    # Reemplazar el clasificador final
    # Original: Linear(576, 1024) → Hardswish → Dropout → Linear(1024, 1000)
    in_features = model.classifier[0].in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.Hardswish(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes),
    )
    
    # Contar parámetros
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params: {total:,}")
    print(f"Trainable: {trainable:,} ({100*trainable/total:.1f}%)")
    
    return model

# Estrategia 1: Feature Extraction (solo clasificador nuevo)
print("=== Feature Extraction ===")
model_frozen = create_mobilenet_transfer(102, freeze=True).to(device)

In [ ]:
def train_model(model, trainloader, valloader, epochs=10, lr=0.001):
    """Training loop genérico."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    history = {'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        # Train
        model.train()
        correct, total = 0, 0
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            _, pred = model(images).max(1)
            total += labels.size(0)
            correct += pred.eq(labels).sum().item()
        train_acc = correct / total
        
        # Validate
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in valloader:
                images, labels = images.to(device), labels.to(device)
                _, pred = model(images).max(1)
                total += labels.size(0)
                correct += pred.eq(labels).sum().item()
        val_acc = correct / total
        
        scheduler.step()
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1:2d}/{epochs} — Train: {train_acc:.3f} — Val: {val_acc:.3f}")
    
    return history

print("\n--- Feature Extraction (capas congeladas) ---")
history_frozen = train_model(model_frozen, trainloader, valloader, epochs=10, lr=0.001)

In [ ]:
# Estrategia 2: Fine-tuning (descongelar últimas capas)
print("=== Fine-tuning ===")
model_finetuned = create_mobilenet_transfer(102, freeze=False).to(device)

# Fine-tuning con LR más bajo para no destruir los pesos pre-entrenados
print("\n--- Fine-tuning (todas las capas, LR bajo) ---")
history_finetuned = train_model(model_finetuned, trainloader, valloader, epochs=10, lr=0.0001)

In [ ]:
# Comparar estrategias
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history_frozen['val_acc'], 'b-o', label='Feature Extraction')
ax.plot(history_finetuned['val_acc'], 'r-o', label='Fine-tuning')
ax.set_title('Comparación de Estrategias de Transfer Learning')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Feature Extraction — Mejor val acc: {max(history_frozen['val_acc']):.3f}")
print(f"Fine-tuning       — Mejor val acc: {max(history_finetuned['val_acc']):.3f}")

## 4. Transfer Learning con TensorFlow/Keras

La misma idea, pero con la API de Keras (más alto nivel).

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")

# Cargar Flowers102 con tf.keras
# (Nota: TF no tiene flowers102 directo, usamos tfds o flowers de tf)
import tensorflow_datasets as tfds

# Si tensorflow_datasets no está disponible, usar CIFAR-10 como alternativa
try:
    ds_train = tfds.load('oxford_flowers102', split='train', as_supervised=True)
    ds_val = tfds.load('oxford_flowers102', split='validation', as_supervised=True)
    NUM_CLASSES = 102
    DATASET_NAME = 'Flowers102'
except Exception:
    (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
    ds_train = tf.data.Dataset.from_tensor_slices((x_train, y_train.flatten()))
    ds_val = tf.data.Dataset.from_tensor_slices((x_test, y_test.flatten()))
    NUM_CLASSES = 10
    DATASET_NAME = 'CIFAR-10'
    print("Usando CIFAR-10 como alternativa")

print(f"Dataset: {DATASET_NAME}")

In [ ]:
# Preprocesamiento
IMG_SIZE = 224

def preprocess_train(image, label):
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.2)
    image = tf.cast(image, tf.float32) / 255.0
    # EfficientNet espera [0, 255], pero aquí normalizamos a [0, 1]
    return image, label

def preprocess_val(image, label):
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

BATCH_SIZE = 32
train_ds = ds_train.map(preprocess_train).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = ds_val.map(preprocess_val).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# EfficientNetV2 con Transfer Learning en Keras
# Keras hace esto MUY fácil

base_model = keras.applications.EfficientNetV2B0(
    include_top=False,       # Sin clasificador final
    weights='imagenet',      # Pesos pre-entrenados
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
)

# Congelar base model
base_model.trainable = False

# Construir modelo completo
model_tf = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax'),
])

model_tf.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model_tf.summary()

print(f"\nTrainable params: {sum(tf.keras.backend.count_params(w) for w in model_tf.trainable_weights):,}")
print(f"Non-trainable: {sum(tf.keras.backend.count_params(w) for w in model_tf.non_trainable_weights):,}")

In [ ]:
# Entrenar en Keras (mucho más simple que PyTorch)
history_tf = model_tf.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
)

print(f"\nMejor val accuracy: {max(history_tf.history['val_accuracy']):.3f}")

In [ ]:
# Fine-tuning en Keras: descongelar últimas capas
base_model.trainable = True

# Congelar las primeras capas (dejar solo las últimas 20 entrenables)
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Recompilar con LR más bajo
model_tf.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# Fine-tune
history_ft_tf = model_tf.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
)

print(f"Fine-tuned val accuracy: {max(history_ft_tf.history['val_accuracy']):.3f}")

## 5. Mejores Prácticas de Transfer Learning

| Situación | Estrategia | Learning Rate |
|-----------|------------|---------------|
| Pocos datos (<500), dominio similar | Feature extraction | 1e-3 |
| Datos moderados (500-5K), similar | Fine-tune últimas capas | 1e-4 |
| Muchos datos (>5K), similar | Fine-tune todo | 1e-5 |
| Dominio muy diferente | Fine-tune todo desde el inicio | 1e-4 |

### Tips:
- **Siempre empezar con feature extraction** — es rápido y da un baseline
- **Data augmentation es crucial** con pocos datos
- **Learning rate bajo** para fine-tuning (no destruir pesos pre-entrenados)
- **Normalización**: usar la misma que el modelo pre-entrenado (ImageNet stats)
- **Batch size más grande** para feature extraction (no necesita muchos gradientes)

## 6. Usando MLForge

Todo lo anterior se puede hacer con un solo comando:

```bash
# Entrenar MobileNetV3 en flowers102 con transfer learning
mlforge train --config config.yaml
```

Donde `config.yaml`:
```yaml
project:
  name: flower_classifier
  task: classification

data:
  source: torchvision
  dataset: flowers102
  input_size: 224

model:
  architecture: mobilenet_v3_small
  pretrained: true          # Transfer learning!
  num_classes: 102

training:
  epochs: 20
  batch_size: 32
  optimizer: adam
  learning_rate: 0.001
  scheduler: cosine
```

MLForge se encarga de todo: descarga del dataset, augmentation,
crear el modelo con transfer learning, training loop, checkpointing, y métricas.

**Siguiente notebook:** Optimización de modelos para edge — quantización, pruning, y cómo reducir modelos de 100MB a 3MB.